# fastsae `hb_area()` engines on an NVIDIA GPU (Kaggle)

This notebook measures `hb_area()` with **INLA**, **JAX on the CPU** and **JAX on the CUDA GPU** (plus
**tipsae**, Stan, for the Beta model) on spatial (Besag ICAR) area-level models. The results feed the
pkgdown article
[hb_area() engines](https://ridsonap.github.io/fastsae/articles/hb-area-engines.html).

It is a **Python** notebook on purpose, because Kaggle gives GPUs to Python notebooks. The notebook
installs R itself and runs everything through `Rscript`.

**How to run**

1. Verify your phone number in your Kaggle account settings. GPU and internet need it.
2. *Create → New Notebook → File → Import Notebook*, then paste the GitHub URL of this file
   (`https://github.com/ridsonap/fastsae/blob/main/benchmarks/kaggle/hb_area_engines_kaggle.ipynb`).
3. In the right-hand panel, *Session options*:
   - **Language: Python**.
   - **Accelerator: GPU T4 x2** (or GPU P100). Check that the panel shows the GPU as on.
   - **Internet: On**.
4. Click **Save Version → Save & Run All (Commit)**. Kaggle runs the notebook on its servers for up
   to 12 hours, without the browser open. Expect about 3–5 hours.
5. When the version is complete, open it and go to the **Output** tab. Download
   `hb_engine_benchmark_kaggle.rds` and `bench.log`.

In [ ]:
# 1. Check the GPU (stops here with a clear message if no GPU is attached)
import shutil, subprocess
if shutil.which("nvidia-smi") is None:
    raise SystemExit("No NVIDIA GPU in this session: set Settings -> Accelerator -> GPU T4 x2 "
                     "(needs a phone-verified account), then run again.")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
%%bash
# 2. R from the CRAN Ubuntu repository, plus the system libraries for sf/spdep and Python venv
set -e
export DEBIAN_FRONTEND=noninteractive
. /etc/os-release
if ! command -v Rscript > /dev/null; then
  apt-get -qq update
  apt-get -qq install -y --no-install-recommends ca-certificates curl gnupg > /dev/null
  curl -fsSL https://cloud.r-project.org/bin/linux/ubuntu/marutter_pubkey.asc \
    | gpg --dearmor -o /usr/share/keyrings/cran.gpg
  echo "deb [signed-by=/usr/share/keyrings/cran.gpg] https://cloud.r-project.org/bin/linux/ubuntu ${VERSION_CODENAME}-cran40/" \
    > /etc/apt/sources.list.d/cran.list
  apt-get -qq update
  apt-get -qq install -y --no-install-recommends r-base r-base-dev > /dev/null
fi
apt-get -qq install -y --no-install-recommends libudunits2-dev libgdal-dev libgeos-dev libproj-dev \
  libcurl4-openssl-dev libssl-dev libxml2-dev libfontconfig1-dev cmake python3-venv > /dev/null
R --version | head -1

In [ ]:
%%writefile /kaggle/working/setup.R
# 3. R packages (binaries from Posit Package Manager), INLA, fastsae, and JAX with CUDA
os <- readLines("/etc/os-release")
codename <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", os, value = TRUE))
options(
  repos = c(P3M = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", codename),
            CRAN = "https://cloud.r-project.org"),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
                          paste(getRversion(), R.version$platform, R.version$arch, R.version$os)),
  Ncpus = parallel::detectCores()
)
have <- function(p) requireNamespace(p, quietly = TRUE)
need <- c("remotes", "sf", "spdep", "reticulate", "ggplot2", "rstan")
if (length(missing <- need[!vapply(need, have, logical(1))]) > 0) install.packages(missing)
if (!have("INLA")) {
  install.packages("INLA", repos = c(getOption("repos"), INLA = "https://inla.r-inla-download.org/R/stable"),
                   dependencies = TRUE)
}
# tipsae separately, keeping its build log: if it fails, the benchmark runs without it
if (!have("tipsae")) install.packages("tipsae", keep_outputs = TRUE)
if (!have("tipsae")) {
  message("tipsae could not be installed; the benchmark will skip it. Last lines of its log:")
  if (file.exists("tipsae.out")) cat(tail(readLines("tipsae.out"), 40), sep = "\n")
}
remotes::install_github("ridsonap/fastsae", upgrade = "never", force = TRUE)
print(sapply(c("fastsae", "INLA", "sf", "spdep", "tipsae", "reticulate"),
             function(p) if (have(p)) as.character(packageVersion(p)) else NA))

Sys.unsetenv("RETICULATE_PYTHON")
fastsae::setup_numpyro_env(device = "cuda")

In [ ]:
%%bash
cd /kaggle/working
Rscript setup.R 2>&1 | tail -40
# Check in a fresh R process (JAX fixes its device per process): it must list a CUDA device
Rscript -e 'fastsae::check_numpyro_available(device = "cuda")'

In [ ]:
%%bash
# 4. Benchmark (INLA, JAX CPU, JAX CUDA, tipsae if installed), in the foreground
cd /kaggle/working
curl -fsSL https://raw.githubusercontent.com/ridsonap/fastsae/main/benchmarks/bench_hb_area_engines.R \
  -o bench_hb_area_engines.R
export RETICULATE_PYTHON=$(Rscript -e 'cat(reticulate::virtualenv_python("r-numpyro-gpu"))')
ENGINES=inla,cpu,cuda
Rscript -e 'quit(status = !requireNamespace("tipsae", quietly = TRUE))' && ENGINES=$ENGINES,tipsae
echo "Engines: $ENGINES"
start=$(date +%s)
Rscript bench_hb_area_engines.R --engines=$ENGINES --out=/kaggle/working/hb_engine_benchmark_kaggle.rds \
  > bench.log 2> bench.err
status=$?
grep -E " n = |Error|Saved|failed" bench.log
[ $status -ne 0 ] && tail -30 bench.err
echo "Exit status: $status   Elapsed: $(( ($(date +%s) - start) / 60 )) min"

In [ ]:
%%bash
# 5. Results (also kept in the saved version of this notebook)
Rscript -e 'res <- readRDS("/kaggle/working/hb_engine_benchmark_kaggle.rds")
str(res$meta[c("machine", "cpu", "cpu_cores", "gpu", "ram_gb", "python")])
print(res$timings); print(res$agreement)'

**6. Download:** open the saved version → **Output** tab → download `hb_engine_benchmark_kaggle.rds`.
This file is added to the article as `inst/extdata/hb_engine_benchmark_kaggle_<gpu>.rds`.